# Phase 1: Complex Data Acquisition & Data Cleaning
### Case Study: PT Andalas Manufaktur — Operation Excellence 2026

**Role:** Data Analytics Team
**Objective of this notebook:**
1. Extract an integrated dataset from `pt_andalas_db` using a multi-table SQL `JOIN`.
2. Profile the raw extract to identify data quality issues.
3. Clean the data (missing values, duplicates, physical anomalies, inconsistent formatting/types).
4. Export an analysis-ready dataset for the next phases of the project.

**Database:** `pt_andalas_db.db` (SQLite, Star Schema)

| Table | Role | Key Columns |
|---|---|---|
| `ms_mesin` | Dimension | id_mesin, nama_mesin, tipe, lokasi |
| `ms_operator` | Dimension | id_operator, nama_lengkap, grup_shift, skill_level |
| `ms_material` | Dimension | id_material, jenis_bahan_baku, vendor_pemasok |
| `tr_produksi` | Fact | tanggal, jam, id_mesin, id_operator, id_material, setting_speed_rpm, suhu_mesin, output_qty_ok, reject_qty_ng |
| `tr_maintenance` | Fact | tanggal, id_mesin, tipe_kerusakan, biaya_perbaikan, durasi_downtime |

> **Note on schema design:** the original brief does not list a foreign key linking `tr_produksi` to `ms_material`. For the star schema to support root-cause analysis by raw material (as required by the project background), `id_material` is included as a foreign key on `tr_produksi`. This is the one modelling assumption made in this notebook.


## 1. Setup & Connect to the Database

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# Automatically select database that contains required tables
possible_db = [
    "pt_andalas_db.db",
    "pt_andalas_db (pengganti uas iad).db"
]

DB_PATH = None

for db in possible_db:
    if Path(db).exists():
        test_conn = sqlite3.connect(db)
        tables = pd.read_sql(
            "SELECT name FROM sqlite_master WHERE type='table';",
            test_conn
        )["name"].tolist()
        test_conn.close()

        if "tr_produksi" in tables and "tr_maintenance" in tables:
            DB_PATH = db
            break

if DB_PATH is None:
    raise FileNotFoundError(
        "Database tidak ditemukan atau tabel tr_produksi/tr_maintenance belum tersedia."
    )

conn = sqlite3.connect(DB_PATH)

print("Connected to:", DB_PATH)
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn))


## 2. Data Acquisition with SQL JOIN

The project instructions require extracting data using a SQL `JOIN` across **at least 3 tables**.
Below, the production fact table `tr_produksi` is joined with all three dimension tables
(`ms_mesin`, `ms_operator`, `ms_material`) — four tables in total — to build a single
denormalized, analysis-ready extract that links **machine + operator + material** for every
production record.

A quick check on the raw fact table shows that `id_mesin` is not consistently written
(e.g. `MSN-04` vs `msn-04`), which would silently break a case-sensitive join and drop those
rows out of the dimension lookup. The join below uses `UPPER()` on both sides of the key so
every transaction still matches its machine, while the inconsistency itself is fixed properly
in the Pandas cleaning step in Section 4.


In [ ]:
# Confirm the id_mesin casing inconsistency exists in the raw transactional table
raw_ids = pd.read_sql("SELECT DISTINCT id_mesin FROM tr_produksi", conn)
print(raw_ids["id_mesin"].tolist())


In [ ]:
query_produksi = '''
SELECT
    p.id_produksi,
    p.tanggal,
    p.jam,
    p.id_mesin,
    m.nama_mesin,
    m.tipe          AS tipe_mesin,
    m.lokasi,
    p.id_operator,
    o.nama_lengkap   AS nama_operator,
    o.grup_shift,
    o.skill_level,
    p.id_material,
    mt.jenis_bahan_baku,
    mt.vendor_pemasok,
    p.setting_speed_rpm,
    p.suhu_mesin,
    p.output_qty_ok,
    p.reject_qty_ng
FROM tr_produksi p
LEFT JOIN ms_mesin    m  ON UPPER(p.id_mesin)    = UPPER(m.id_mesin)
LEFT JOIN ms_operator o  ON p.id_operator        = o.id_operator
LEFT JOIN ms_material mt ON p.id_material        = mt.id_material
'''

df_raw = pd.read_sql(query_produksi, conn)
print("Raw extract shape:", df_raw.shape)
df_raw.head(10)


A second extract joins the maintenance fact table with the machine dimension. This will support
the failure/downtime analysis in later phases of the project.


In [ ]:
query_maintenance = '''
SELECT
    mn.id_maintenance,
    mn.tanggal,
    mn.id_mesin,
    m.nama_mesin,
    m.lokasi,
    mn.tipe_kerusakan,
    mn.biaya_perbaikan,
    mn.durasi_downtime
FROM tr_maintenance mn
LEFT JOIN ms_mesin m ON mn.id_mesin = m.id_mesin
'''

df_maint_raw = pd.read_sql(query_maintenance, conn)
print("Raw maintenance extract shape:", df_maint_raw.shape)
df_maint_raw.head(10)


## 3. Data Profiling — Identifying Quality Issues

Before cleaning anything, profile the raw extract to confirm and quantify the issues flagged in
the project brief: missing values, duplicate rows, physical anomalies in temperature, and
inconsistent text/number/date formatting.


In [ ]:
print("=== dtypes ===")
print(df_raw.dtypes)
print()
print("=== Missing values per column ===")
print(df_raw.replace("", np.nan).isnull().sum())


In [ ]:
print("Exact duplicate rows in df_raw:", df_raw.duplicated().sum())
df_raw[df_raw.duplicated(keep=False)].sort_values("id_produksi").head()


In [ ]:
# Inspect distinct raw text values that should represent the same category
print("Distinct 'lokasi' values (machine dimension):")
print(df_raw["lokasi"].unique())
print()
print("Distinct 'grup_shift' values (operator dimension):")
print(df_raw["grup_shift"].unique())
print()
print("Distinct 'jenis_bahan_baku' values (material dimension):")
print(df_raw["jenis_bahan_baku"].unique())


In [ ]:
# suhu_mesin (machine temperature) is stored as TEXT and mixes valid readings with
# physically impossible values (negative, or implausibly high sensor-fault spikes)
suhu_numeric = pd.to_numeric(df_raw["suhu_mesin"].replace("", np.nan), errors="coerce")
print(suhu_numeric.describe())
print()
print("Negative readings:", (suhu_numeric < 0).sum())
print("Implausibly high readings (>150C):", (suhu_numeric > 150).sum())


In [ ]:
# setting_speed_rpm mixes plain numbers with strings like "1500 rpm"
print("Sample raw speed values:", df_raw['setting_speed_rpm'].unique()[:10])
print()
# tanggal mixes at least 3 different date formats
print("Sample raw date values:", df_raw['tanggal'].unique()[:10])


## 4. Data Cleaning

Cleaning is applied step by step, in this order:
1. Remove exact duplicate rows.
2. Standardize text fields (casing, whitespace).
3. Standardize the machine ID key used for joining.
4. Parse mixed date formats into a single `datetime` type.
5. Parse mixed-format numeric fields (`setting_speed_rpm`) into clean numbers.
6. Treat physically impossible temperature readings as invalid, then impute.
7. Handle missing values in `output_qty_ok`.
8. Final type casting and validation.


In [ ]:
df = df_raw.copy()

# 4.1 Remove exact duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} exact duplicate rows. New shape: {df.shape}")


In [ ]:
# 4.2 Standardize text fields: trim whitespace, collapse double spaces, normalize case
text_cols = ["nama_mesin", "lokasi", "tipe_mesin", "nama_operator",
             "jenis_bahan_baku", "vendor_pemasok"]

for c in text_cols:
    df[c] = (
        df[c]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

# str.title() turns the "PT" company-name prefix into "Pt" — restore it
df["vendor_pemasok"] = df["vendor_pemasok"].str.replace(r"^Pt\b", "PT", regex=True)

# grup_shift: collapse to a single uppercase letter (A / B / C), regardless of
# whether the raw value was "A", "Shift A", "shift b", or " B "
df["grup_shift"] = (
    df["grup_shift"].astype(str).str.strip().str.upper().str.extract(r"([ABC])")[0]
)

print(df["lokasi"].unique())
print(df["grup_shift"].unique())
print(df["jenis_bahan_baku"].unique())


In [ ]:
# 4.3 Standardize id_mesin casing so it always matches the dimension key format (e.g. MSN-01)
df["id_mesin"] = df["id_mesin"].str.upper()
print(df["id_mesin"].unique())


In [ ]:
# 4.4 Parse mixed date formats (YYYY-MM-DD, DD/MM/YYYY, DD-Mon-YY) into a single datetime column
def parse_mixed_date(value):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%b-%y"):
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df["tanggal"] = df["tanggal"].apply(parse_mixed_date)
print("Unparsed dates after standardization:", df["tanggal"].isna().sum())
df[["tanggal"]].head()


In [ ]:
# 4.5 Clean setting_speed_rpm: strip the unit suffix and cast to numeric
df["setting_speed_rpm"] = (
    df["setting_speed_rpm"]
    .astype(str)
    .str.replace("rpm", "", case=False, regex=False)
    .str.strip()
)
df["setting_speed_rpm"] = pd.to_numeric(df["setting_speed_rpm"], errors="coerce")
df["setting_speed_rpm"].describe()


In [ ]:
# 4.6 Handle physically impossible temperature readings
# Plausible operating range for the stamping process is assumed to be 40C - 120C.
# Out-of-range or missing readings are treated as invalid, then imputed using the
# median temperature for the same machine (more representative than a global median).
df["suhu_mesin"] = pd.to_numeric(df["suhu_mesin"].replace("", np.nan), errors="coerce")

invalid_mask = (df["suhu_mesin"] < 40) | (df["suhu_mesin"] > 120)
print("Invalid/out-of-range temperature readings flagged:", invalid_mask.sum())

df.loc[invalid_mask, "suhu_mesin"] = np.nan

df["suhu_mesin"] = df.groupby("id_mesin")["suhu_mesin"].transform(
    lambda s: s.fillna(s.median())
)
print("Remaining missing temperature values after imputation:", df["suhu_mesin"].isna().sum())
df["suhu_mesin"].describe()


In [ ]:
# 4.7 Handle missing values in output_qty_ok
df["output_qty_ok"] = pd.to_numeric(df["output_qty_ok"].replace("", np.nan), errors="coerce")
df["reject_qty_ng"] = pd.to_numeric(df["reject_qty_ng"].replace("", np.nan), errors="coerce")

missing_output = df["output_qty_ok"].isna().sum()
print("Missing output_qty_ok before imputation:", missing_output)

# Impute using the median output for the same machine, since output volume is
# largely a function of machine capacity/cycle time
df["output_qty_ok"] = df.groupby("id_mesin")["output_qty_ok"].transform(
    lambda s: s.fillna(s.median())
)
df["output_qty_ok"] = df["output_qty_ok"].round().astype(int)
df["reject_qty_ng"] = df["reject_qty_ng"].fillna(0).astype(int)

print("Missing output_qty_ok after imputation:", df["output_qty_ok"].isna().sum())


In [ ]:
# 4.8 skill_level: cast to numeric, impute missing with the operator group's median skill level
df["skill_level"] = pd.to_numeric(df["skill_level"], errors="coerce")
df["skill_level"] = df["skill_level"].fillna(df["skill_level"].median()).astype(int)

df[["nama_operator", "grup_shift", "skill_level"]].drop_duplicates().head()


## 5. Clean the Maintenance Extract

In [ ]:
df_maint = df_maint_raw.copy()

# Standardize machine id and free-text fault type
df_maint["id_mesin"] = df_maint["id_mesin"].str.upper()
df_maint["tipe_kerusakan"] = df_maint["tipe_kerusakan"].str.strip().str.title()

# nama_mesin / lokasi are inherited from the same ms_mesin dimension as the production
# extract, so apply the same whitespace/casing standardization for consistency
for c in ["nama_mesin", "lokasi"]:
    df_maint[c] = (
        df_maint[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True).str.title()
    )

# Parse mixed date formats (reuse the same helper as the production extract)
df_maint["tanggal"] = df_maint["tanggal"].apply(parse_mixed_date)

# Clean biaya_perbaikan (repair cost): strip "Rp", thousands separators ("." or ","), then cast to numeric
df_maint["biaya_perbaikan"] = (
    df_maint["biaya_perbaikan"]
    .astype(str)
    .str.replace("Rp", "", case=False, regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df_maint["biaya_perbaikan"] = pd.to_numeric(df_maint["biaya_perbaikan"], errors="coerce")
# Missing repair cost imputed with the median cost for the same fault type
df_maint["biaya_perbaikan"] = df_maint.groupby("tipe_kerusakan")["biaya_perbaikan"].transform(
    lambda s: s.fillna(s.median())
)

# Standardize durasi_downtime to a single numeric unit: minutes
def parse_downtime_to_minutes(value):
    value = str(value).strip().lower()
    if "jam" in value:
        hours = float(value.replace("jam", "").strip())
        return hours * 60
    if "menit" in value:
        return float(value.replace("menit", "").strip())
    return pd.to_numeric(value, errors="coerce")

df_maint["durasi_downtime_menit"] = df_maint["durasi_downtime"].apply(parse_downtime_to_minutes)
df_maint = df_maint.drop(columns=["durasi_downtime"])

print("Missing values remaining:")
print(df_maint.isna().sum())
df_maint.head(10)


## 6. Final Validation

Re-run the profiling checks from Section 3 against the cleaned datasets to confirm every
identified issue has been resolved.


In [ ]:
print("=== df (production) — final check ===")
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:\n", df.isnull().sum())
print()
print("Temperature range: {:.1f} to {:.1f} C".format(df['suhu_mesin'].min(), df['suhu_mesin'].max()))
print("id_mesin distinct values:", sorted(df['id_mesin'].unique()))
print("lokasi distinct values:", sorted(df['lokasi'].unique()))


In [ ]:
print("=== df_maint (maintenance) — final check ===")
print("Shape:", df_maint.shape)
print("Duplicate rows:", df_maint.duplicated().sum())
print("Missing values:\n", df_maint.isnull().sum())


In [ ]:
df.dtypes


## 7. Export Cleaned Datasets

The cleaned, analysis-ready extracts are exported as CSV files. These will be the input for
Phase 2 (root-cause analysis / diagnostics) and Phase 3 (failure prediction) of the project.


In [ ]:
df.to_csv("clean_tr_produksi.csv", index=False)
df_maint.to_csv("clean_tr_maintenance.csv", index=False)

print("Exported:")
print(" - clean_tr_produksi.csv   ", df.shape)
print(" - clean_tr_maintenance.csv", df_maint.shape)


In [ ]:
conn.close()


## 8. Summary of Phase 1

| Issue identified in raw extract | Cleaning action taken |
|---|---|
| Exact duplicate transaction rows | Removed with `drop_duplicates()` |
| Inconsistent text casing/spacing (`lokasi`, `grup_shift`, `jenis_bahan_baku`, machine/operator names) | Trimmed whitespace, collapsed repeated spaces, standardized casing |
| Inconsistent `id_mesin` key casing | Standardized to uppercase to match the dimension table key |
| Three mixed date formats in `tanggal` | Parsed into a single `datetime` column |
| Mixed-format `setting_speed_rpm` (e.g. "1500 rpm" vs `1500`) | Unit suffix stripped, cast to numeric |
| Physically impossible `suhu_mesin` readings (negative values, 999C sensor spikes) | Flagged as invalid (outside 40-120C), then imputed using the per-machine median |
| Missing `output_qty_ok` values | Imputed using the per-machine median output |
| Missing `skill_level` values | Imputed using the median skill level |
| Mixed currency formatting in `biaya_perbaikan` ("Rp 1.500.000" vs "1,500,000" vs `1500000`) | Currency symbols/separators stripped, cast to numeric |
| Mixed time units in `durasi_downtime` ("2 jam" vs "120 menit") | Standardized to a single numeric column in minutes |

**Deliverable:** this notebook, plus two cleaned CSV extracts (`clean_tr_produksi.csv`,
`clean_tr_maintenance.csv`) ready for the diagnostic and predictive phases of the
Operation Excellence 2026 initiative.
